# D1-03 Manual LCA with NumPy matrices (Heijungs and Suh 2002) vs Brightway

This notebook introduces the conventional matrix computation structure
(A, B, Q, f) and compares the manual result with Brightway's result.
            


## Learning goals

- Build a tiny LCA model with NumPy matrices.
- Compute scaling vector, inventory, and impact score manually.
- Implement the same toy model in Brightway and compare outputs.
            


In [ ]:
import numpy as np

# A: technosphere matrix
# rows: products [P1, P2], columns: activities [P1, P2]
A = np.array([
    [1.0,  0.0],
    [-0.5, 1.0],
])

# B: biosphere matrix
# rows: elementary flows [CO2], columns: activities [P1, P2]
B = np.array([[1.5, 0.2]])

# Q: characterization matrix (single category)
Q = np.array([[1.0]])

# Functional unit demand vector (1 unit of product P1)
f = np.array([1.0, 0.0])

x = np.linalg.solve(A, f)
g = B @ x
h = Q @ g

print('Scaling vector x:', x)
print('Inventory g:', g)
print('Impact h:', h)
            


For this system, the expected impact is 1.6 impact units.
            


In [ ]:
import bw2data as bd
import bw2calc as bc

bd.projects.set_current('barcelona-d1-matrix-bridge')

biosphere_name = 'biosphere_toy'
tech_name = 'toy_technosphere'
method_key = ('Toy method', 'climate change', 'GWP100')

bd.Database(biosphere_name).write({
    (biosphere_name, 'co2'): {
        'name': 'Carbon dioxide, fossil',
        'unit': 'kg',
        'type': 'emission',
    }
})

bd.Database(tech_name).write({
    (tech_name, 'P1'): {
        'name': 'Toy process P1',
        'unit': 'kg',
        'exchanges': [
            {'input': (tech_name, 'P1'), 'amount': 1.0, 'type': 'production'},
            {'input': (tech_name, 'P2'), 'amount': 0.5, 'type': 'technosphere'},
            {'input': (biosphere_name, 'co2'), 'amount': 1.5, 'type': 'biosphere'},
        ],
    },
    (tech_name, 'P2'): {
        'name': 'Toy process P2',
        'unit': 'kg',
        'exchanges': [
            {'input': (tech_name, 'P2'), 'amount': 1.0, 'type': 'production'},
            {'input': (biosphere_name, 'co2'), 'amount': 0.2, 'type': 'biosphere'},
        ],
    },
})

if method_key not in bd.methods:
    bd.Method(method_key).register(unit='kg CO2-eq')

bd.Method(method_key).write([
    ((biosphere_name, 'co2'), 1.0)
])

p1 = bd.Database(tech_name).get('P1')
lca = bc.LCA({p1: 1.0}, method_key)
lca.lci()
lca.lcia()

print('Brightway impact score:', lca.score)
            


In [ ]:
manual_score = float(h.ravel()[0])
bw_score = float(lca.score)

print('Manual score:', manual_score)
print('Brightway score:', bw_score)
print('Absolute difference:', abs(manual_score - bw_score))
assert np.isclose(manual_score, bw_score), 'Scores do not match'
            


## Exercise

Change the functional unit to 2 units of P1 and compare manual and Brightway scores.
            


In [ ]:
# TODO
            


In [ ]:
f2 = np.array([2.0, 0.0])
x2 = np.linalg.solve(A, f2)
g2 = B @ x2
h2 = Q @ g2
manual_score_2 = float(h2.ravel()[0])

lca2 = bc.LCA({p1: 2.0}, method_key)
lca2.lci()
lca2.lcia()

print('Manual score (FU=2):', manual_score_2)
print('Brightway score (FU=2):', float(lca2.score))
assert np.isclose(manual_score_2, lca2.score)
            
